# RAG Foundation Lab

**課程**: Session 1 - Technical Foundations  
**目標**: 理解向量搜尋與 RAG 原理，體驗語義搜尋的威力

---

## 📚 Lab 結構

- **Part 1**: 向量搜尋 vs 關鍵字搜尋
- **Part 2**: 簡易 RAG 實作

---

## Part 1: 向量搜尋 vs 關鍵字搜尋

### 🎯 目標

- 理解向量搜尋與關鍵字搜尋的差異
- 體驗語義理解的威力
- 理解為什麼 RAG 需要向量搜尋

---

### Task 1.1: 環境設定與準備知識庫

**情境**: 你正在建立一個程式碼知識庫，幫助團隊快速找到相關文件

In [ ]:
# 安裝必要的套件
!pip install sentence-transformers scikit-learn numpy -q

print("✓ 套件安裝完成！")

In [ ]:
# 匯入必要的函式庫
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 載入 Embedding 模型
print("載入 Embedding 模型中...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✓ 模型載入完成！\n")

In [ ]:
# 準備程式碼知識庫（模擬真實文件）
knowledge_base = [
    {
        "id": "doc1",
        "title": "Authentication API",
        "content": "Our REST API uses JWT tokens for user authentication. Send the token in the Authorization header."
    },
    {
        "id": "doc2",
        "title": "Database Schema",
        "content": "The users table contains columns: id, email, password_hash, created_at, updated_at."
    },
    {
        "id": "doc3",
        "title": "Error Handling",
        "content": "All API endpoints return standardized error responses with status code and message."
    },
    {
        "id": "doc4",
        "title": "Rate Limiting",
        "content": "API requests are limited to 100 calls per minute per user to prevent abuse."
    },
    {
        "id": "doc5",
        "title": "Webhook Setup",
        "content": "Configure webhooks to receive real-time notifications when events occur in the system."
    },
    {
        "id": "doc6",
        "title": "User Login Flow",
        "content": "Users sign in with email and password. The system validates credentials and returns a session token."
    },
    {
        "id": "doc7",
        "title": "API Throttling",
        "content": "Requests exceeding the rate limit will receive a 429 Too Many Requests response."
    },
    {
        "id": "doc8",
        "title": "Security Best Practices",
        "content": "Always use HTTPS, validate input, sanitize output, and implement proper authentication."
    }
]

print(f"✓ 知識庫準備完成，共 {len(knowledge_base)} 份文件")
print("\n範例文件:")
print(f"  {knowledge_base[0]['title']}: {knowledge_base[0]['content'][:50]}...")

---

### Task 1.2: 關鍵字搜尋實作

**方法**: 簡單的字串匹配

In [ ]:
import re

def keyword_search(query, knowledge_base, top_k=3):
    """
    關鍵字搜尋：簡單的字串匹配
    """
    query_lower = query.lower()
    results = []

    for doc in knowledge_base:
        # 計算關鍵字出現次數
        content_lower = doc['content'].lower()
        title_lower = doc['title'].lower()

        # 簡單計分：標題匹配 + 2 分，內容匹配 + 1 分
        score = 0
        for word in query_lower.split():
            score += len(re.findall(r'\b' + re.escape(word) + r'\b', title_lower)) * 2
            score += len(re.findall(r'\b' + re.escape(word) + r'\b', content_lower)) * 1

        if score > 0:
            results.append({
                'doc': doc,
                'score': score
            })

    # 排序並返回 top_k
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

def show_search_results(query, knowledge_base, top_k=3):
    print("\n" + "=" * 80)
    print(f"🔍 關鍵字搜尋：'{query}'\n")

    results = keyword_search(query, knowledge_base, top_k=3)

    if results:
        for i, result in enumerate(results, 1):
            doc = result['doc']
            print(f"\n第 {i} 名 (分數: {result['score']}):")
            print(f"  標題: {doc['title']}")
            print(f"  內容: {doc['content']}")
    else:
        print("❌ 沒有找到相關文件")

print("✓ 關鍵字搜尋函式準備完成")

In [ ]:
# 測試關鍵字搜尋
query1 = "authentication"
query2 = "How do I authenticate users?"

show_search_results(query1, knowledge_base)
show_search_results(query2, knowledge_base)

### 📊 觀察結果

**關鍵字搜尋的問題**:
- 只能匹配「完全相同」的字詞
- "authenticate" 和 "authentication" 被視為不同詞
- 無法理解同義詞（如 "sign in" vs "login"）

---

### Task 1.3: 向量搜尋實作

**方法**: 使用 Embedding 和語義相似度

In [ ]:
# 步驟 1: 為知識庫的每份文件建立向量
print("建立知識庫向量索引...")

for doc in knowledge_base:
    # 將標題和內容合併後轉換為向量
    text = f"{doc['title']}. {doc['content']}"
    doc['embedding'] = model.encode(text)

print("✓ 向量索引建立完成！")
print(f"  每份文件的向量維度: {len(knowledge_base[0]['embedding'])}")

In [ ]:
def vector_search(query, knowledge_base, top_k=3):
    """
    向量搜尋：使用語義相似度
    """
    # 步驟 1: 將查詢轉換為向量
    query_embedding = model.encode(query)

    # 步驟 2: 計算查詢與所有文件的相似度
    results = []
    for doc in knowledge_base:
        similarity = cosine_similarity(
            [query_embedding],
            [doc['embedding']]
        )[0][0]

        results.append({
            'doc': doc,
            'score': similarity
        })

    # 步驟 3: 排序並返回最相關的 top_k 份文件
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

print("✓ 向量搜尋函式準備完成")

In [ ]:
# 測試向量搜尋（使用相同的查詢）
query1 = "How do I authenticate users?"

print(f"🔍 向量搜尋：'{query1}'\n")
print("=" * 80)

results = vector_search(query1, knowledge_base, top_k=3)

for i, result in enumerate(results, 1):
    doc = result['doc']
    print(f"\n第 {i} 名 (相似度: {result['score']:.3f}):")
    print(f"  標題: {doc['title']}")
    print(f"  內容: {doc['content']}")

print("\n" + "=" * 80)

### 📊 觀察結果

**向量搜尋的優勢**:
- ✅ 理解語義："authenticate" 能找到 "authentication" 相關文件
- ✅ 同義詞："sign in", "login", "authenticate" 都能找到相同文件
- ✅ 相似度分數：量化文件相關程度

---

### Task 1.4: 對比實驗

**挑戰**: 用不同的查詢方式測試兩種搜尋

In [ ]:
# 測試案例 1: 同義詞挑戰
test_queries = [
    "How do I authenticate users?", # 原始查詢
    "What's the sign in process?", # 同義詞
    "How to handle too many requests?", # 語義相關
    "我如何限制同一裝置的呼叫次數？" # 跨語言
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"查詢: '{query}'\n")

    # 關鍵字搜尋
    print("🔤 關鍵字搜尋結果:")
    keyword_results = keyword_search(query, knowledge_base, top_k=1)
    if keyword_results:
        print(f"  ✓ {keyword_results[0]['doc']['title']} (分數: {keyword_results[0]['score']})")
    else:
        print("  ❌ 沒有找到相關文件")

    # 向量搜尋
    print("\n🔮 向量搜尋結果:")
    vector_results = vector_search(query, knowledge_base, top_k=1)
    print(f"  ✓ {vector_results[0]['doc']['title']} (相似度: {vector_results[0]['score']:.3f})")

print(f"\n{'='*80}")

### 💡 關鍵洞察

**關鍵字搜尋**:
- 🚫 同義詞問題："sign in" 找不到 "authentication"
- 🚫 語言限制：中文查詢找不到英文文件
- ✅ 精確匹配：如果用詞完全相同，速度快

**向量搜尋**:
- ✅ 語義理解：理解不同用詞的相同意思
- ✅ 跨語言：中文查詢能找到英文文件
- ✅ 智能排序：按語義相關度排序
- ⚠️ 計算成本：需要建立向量索引

---

### Task 1.5: 討論與實驗

⚠️ 已知 `all-MiniLM-L6-v2` 模型無法正確識別中文

模型 `all-MiniLM-L6-v2` 與 `paraphrase-multilingual-MiniLM-L12-v2`

對於: 'How do I authenticate users?' 是否有差異?

In [ ]:
# 載入 all-MiniLM-L6-v2 模型
print("載入 all-MiniLM-L6-v2 模型中...")
model_general = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ 模型載入完成！\n")

In [ ]:
import copy

# 建立新的知識庫: knowledge_base_general
print("建立 knowledge_base_general...")
knowledge_base_general = copy.deepcopy(knowledge_base)
for doc in knowledge_base_general:
    text = f"{doc['title']}. {doc['content']}"
    # 使用另一個模型: all-MiniLM-L6-v2
    doc['embedding'] = model_general.encode(text)

print("✓ knowledge_base_general 建立完成！")
print(f"  每份文件的向量維度: {len(knowledge_base_general[0]['embedding'])}")

In [ ]:
# 測試不同模型
your_query = "How do I authenticate users?"

# all-MiniLM-L6-v2
print("使用 all-MiniLM-L6-v2 產生的向量知識庫")
print(f"🔍 搜尋：'{your_query}'\n")
print("=" * 80)

results = vector_search(your_query, knowledge_base_general, top_k=3)

for i, result in enumerate(results, 1):
    doc = result['doc']
    print(f"\n第 {i} 名 (相似度: {result['score']:.3f}):")
    print(f"  標題: {doc['title']}")
    print(f"  內容: {doc['content']}")

print("\n" + "=" * 80)

# paraphrase-multilingual-MiniLM-L12-v2
print("使用 paraphrase-multilingual-MiniLM-L12-v2 產生的向量知識庫")
print(f"🔍 搜尋：'{your_query}'\n")
print("=" * 80)

results = vector_search(your_query, knowledge_base, top_k=3)

for i, result in enumerate(results, 1):
    doc = result['doc']
    print(f"\n第 {i} 名 (相似度: {result['score']:.3f}):")
    print(f"  標題: {doc['title']}")
    print(f"  內容: {doc['content']}")

print("\n" + "=" * 80)

❓ 為什麼結果不同？

### ✅ Part 1 完成檢查點

**你學到了**:
- ✓ 關鍵字搜尋 vs 向量搜尋的差異
- ✓ 向量搜尋能理解同義詞和跨語言
- ✓ 語義相似度比字串匹配更符合實際應用情境

---

## Part 2: 簡易 RAG 實作

### 🎯 學習目標

- 理解 RAG 的三個階段：Retrieval → Augmented → Generation
- 實作完整的 RAG 流程
- 體驗 NotebookLM 背後的原理

---

### Task 2.1: 理解 RAG 流程

**RAG 三階段**:

```
用戶問題
   ↓
1️⃣ Retrieval (檢索)
   └→ 使用向量搜尋找到相關文件
   ↓
2️⃣ Augmented (增強)
   └→ 將找到的文件加入 Prompt
   ↓
3️⃣ Generation (生成)
   └→ LLM 基於檢索到的知識生成回答
```

In [ ]:
def simple_rag(query, knowledge_base, top_k=2):
    """
    簡易 RAG 實作（不使用真實 LLM，用模擬回應）
    """
    print(f"問題: {query}\n")
    print("=" * 80)

    # 階段 1️⃣: Retrieval (檢索)
    print("\n階段 1️⃣: Retrieval (檢索)")
    print("-" * 80)
    results = vector_search(query, knowledge_base, top_k=top_k)

    print(f"✓ 從知識庫中找到 {len(results)} 份相關文件:\n")
    retrieved_docs = []
    for i, result in enumerate(results, 1):
        doc = result['doc']
        retrieved_docs.append(doc)
        print(f"文件 {i} (相似度: {result['score']:.3f}):")
        print(f"  標題: {doc['title']}")
        print(f"  內容: {doc['content']}\n")

    # 階段 2️⃣: Augmented (增強)
    print("\n階段 2️⃣: Augmented (增強)")
    print("-" * 80)

    # 建立增強的 Prompt
    context = "\n\n".join([
        f"文件: {doc['title']}\n內容: {doc['content']}"
        for doc in retrieved_docs
    ])

    augmented_prompt = f"""根據以下文件回答問題。如果文件中沒有相關資訊，請誠實說不知道。

相關文件:
{context}

問題: {query}"""

    print("✓ 建立增強的 Prompt:\n")
    print(augmented_prompt)

    # 階段 3️⃣: Generation (生成)
    print("\n" + "=" * 80)
    print("\n階段 3️⃣: Generation (生成)")
    print("-" * 80)
    print("\n💡 在實際的 RAG 系統中，這個 Prompt 會送給 LLM (如 GPT-4, Claude)")
    print("   LLM 會基於檢索到的文件生成準確的回答")
    print("\n📌 NotebookLM 就是這樣運作的！")

    return augmented_prompt

print("✓ RAG 函式準備完成")

In [ ]:
# 測試 RAG 流程
query = "How does authentication work in our API?"

augmented_prompt = simple_rag(query, knowledge_base, top_k=2)


### Task 2.2: 如果直接詢問 LLM

---

**情境 1: 直接詢問 LLM**

問題: How does authentication work in our API?

#### ❌ 問題: LLM 不知道你的 API 如何運作

可能會：
- 編造答案（幻覺）
- 給出通用建議（不符合你的實際情況）
- 說不知道

---

**情境 2: 使用 RAG**

#### ✅ 優點: LLM 基於你的實際文件回答

- **準確**：基於真實的技術文件
- **可追溯**：可以引用來源
- **私有知識**：你的專案特定資訊

---


---

### Task 2.3: 實務應用場景

**RAG 在實務中的應用**

In [ ]:
# 多種查詢類型測試
test_scenarios = [
    {
        "scenario": "新人 Onboarding",
        "query": "What are the security best practices I should follow?"
    },
    {
        "scenario": "API 整合",
        "query": "How do I prevent getting rate limited?"
    },
    {
        "scenario": "故障排查",
        "query": "What does a 429 error mean?"
    }
]

for scenario_data in test_scenarios:
    print("\n" + "=" * 80)
    print(f"\n應用場景: {scenario_data['scenario']}")
    print("-" * 80)

    query = scenario_data['query']
    print(f"問題: {query}\n")

    # 使用 RAG 找相關文件
    results = vector_search(query, knowledge_base, top_k=2)

    print("RAG 找到的相關文件:")
    for i, result in enumerate(results, 1):
        doc = result['doc']
        print(f"\n  {i}. {doc['title']} (相似度: {result['score']:.3f})")
        print(f"     {doc['content'][:80]}...")

    print(f"\n💡 RAG 能快速找到相關文件，幫助 {scenario_data['scenario']}")

### 💡 RAG 實務應用總結

1. **NotebookLM**（Google）
   - 上傳文件 → 建立向量索引
   - 提問 → RAG 檢索 → 生成回答
   - 引用來源 → 可追溯

2. **Cursor @codebase**
   - 專案程式碼 → 向量索引
   - "這個 bug 在哪裡？" → 找相關檔案
   - 基於實際程式碼生成解答

3. **內部知識庫**
   - 內部文件、Wiki、Code
   - 員工提問 → RAG 系統
   - 減少對人的重複問答，提升效率


---

### ✅ Part 2 完成檢查點

**你學到了**:
- ✓ RAG 三階段：Retrieval → Augmented → Generation
- ✓ RAG 如何解決 AI 的知識限制
- ✓ NotebookLM 背後的運作原理
- ✓ RAG 應用場景

---

## ➡️ 下一步

Prompt Engineering 🔥
